# Датасет k_VV и регрессия по температуре

1. **Датасет** — длинный формат: `T`, `i1`, `f1`, `i2`, `f2`, `k_VV` (см³/с, как в `k_vv_mm`). Имя файла: `{particle1}_{particle2}_dataset.csv`.
2. **Регрессия** — модель в логарифме: $\ln k(T) = \sum_{j=0}^{n-1} a_j\, T^{-j/3}$; для каждого перехода подбирается степень полинома с минимальной MAPE (как в `regression.ipynb`). Результат: `{particle1}_{particle2}_regression_coefs.csv` со столбцами `i1`, `f1`, `i2`, `f2`, `a_0`, …, `a_{n-1}` (недостающие коэффициенты — пустые ячейки).
3. **График** — сравнение «точного» `k_vv_mm` и значения по регрессии для выбранного перехода.

**Замечание.** Для регрессии нужна **сетка температур** (не одна точка). Ниже по умолчанию диапазон начинается с 300 K; при необходимости оставьте только `np.array([300.0])` для выгрузки датасета без подгонки.

Тетрадку следует запускать с рабочей папкой `FHO_FR_VV` (импорты `particles_data`, `k_vv_mm`).

Тетрадку следует запускать с рабочей папкой `FHO_FR_VV` (импорты `particles_data`, `k_vv_mm`).

In [ ]:
from __future__ import annotations

import multiprocessing as mp
import os
import sys
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from particles_data import O2, N2
from k_vv_mm import k_vv_mm

## Параметры

In [ ]:
# Имена — только для имён файлов (объект O2 задаёт данные модели)
# PARTICLE1_NAME = "O2"
# PARTICLE2_NAME = "O2"
PARTICLE1_NAME = "N2"
PARTICLE2_NAME = "N2"

# M1, M2 = O2, O2
M1, M2 = N2, N2
# Максимальные колебательные уровни 0 … V_MAX включительно (как в R_VV / coeffs_sparse)
V_MAX = 4

# Сетка температур для датасета и регрессии [K]
TEMPERATURES_K = np.linspace(300.0, 6000.0, 10)

WORKDIR = Path(".")
DATASET_PATH = WORKDIR / f"{PARTICLE1_NAME}_{PARTICLE2_NAME}_dataset_1.csv"
# Переходы, которые вызывает k_lookup внутри R_VV_fast (все четыре ветки, v и v_ по 0…V_MAX)
DATASET_RVV_FAST_PATH = (
    WORKDIR / f"{PARTICLE1_NAME}_{PARTICLE2_NAME}_dataset_rvv_fast_1.csv"
)
COEFS_PATH = WORKDIR / f"{PARTICLE1_NAME}_{PARTICLE2_NAME}_regression_coefs_1.csv"

# Подбор степени полинома для ln k (число коэффицициентов = степень + 1)
POLY_TERM_COUNTS = range(4, 8)

# Параллельная генерация dataset_rvv_fast (процессы); None = все доступные ядра
RVV_FAST_WORKERS = os.cpu_count()

# Переход для визуальной проверки графика (начальное → конечное по молекулам 1 и 2)


## Генерация датасета

In [ ]:
# Датасет: переходы с i2=i1+1, f2=f1−1 при i1+1≤V_MAX, f1≥1, i1>f1 (длинный формат как в разделе ниже).
rows = []
t0 = time.time()
for T in TEMPERATURES_K:
    Tf = float(T)
    for i1 in range(V_MAX):
        for f1 in range(V_MAX):
            print(f"i1={i1} f1={f1}")
            if i1 + 1 <= V_MAX and f1 - 1 >= 0 and i1 > f1:
                i2, f2 = i1 + 1, f1 - 1
                print(f"T={T} i1={i1} f1={f1} i2={i2} f2={f2}")
                k = float(k_vv_mm(M1, M2, i1, f1, i2, f2, Tf))
                rows.append({"T": Tf, "i1": i1, "f1": f1, "i2": i2, "f2": f2, "k_VV": k})

df_dataset = pd.DataFrame(rows)
elapsed = time.time() - t0
n_tr = df_dataset.drop_duplicates(subset=["i1", "f1", "i2", "f2"]).shape[0]
print(f"Уникальных переходов: {n_tr}, строк: {len(df_dataset)}, время: {elapsed:.2f} s")
df_dataset.to_csv(DATASET_PATH, index=False)
print(f"Сохранено: {DATASET_PATH.resolve()}")
df_dataset.head()

### Датасет для `R_VV_fast`

Те же столбцы `T`, `i1`, `f1`, `i2`, `f2`, `k_VV`, но только для четырёх семейств вызовов `k_lookup` из `R_VV_fast` (`k1`…`k4`) при `v`, `v_` ∈ {0,…,`V_MAX`} и тех же `if`, что в модели. Файл: `DATASET_RVV_FAST_PATH`.

In [ ]:
def transitions_for_r_vv_fast(v_max: int) -> set[tuple[int, int, int, int]]:
    """Уникальные (i1, f1, i2, f2), как в R_VV_fast при переборе v, v_ по 0…v_max."""
    out: set[tuple[int, int, int, int]] = set()
    for v in range(v_max + 1):
        for v_ in range(v_max + 1):
            if v + 1 <= v_max and v_ + 1 <= v_max:
                out.add((v_, v_ + 1, v + 1, v))
            if v - 1 >= 0 and v_ - 1 >= 0:
                out.add((v_, v_ - 1, v - 1, v))
            if v - 1 >= 0 and v_ + 1 <= v_max:
                out.add((v_, v_ + 1, v, v - 1))
            if v + 1 <= v_max and v_ - 1 >= 0:
                out.add((v_, v_ - 1, v, v + 1))
    return out


_M1_RVV = None
_M2_RVV = None


def _rvv_worker_init(m1, m2) -> None:
    global _M1_RVV, _M2_RVV
    _M1_RVV, _M2_RVV = m1, m2


def _compute_rvv_row(task: tuple[int, int, int, int, float]) -> dict:
    i1, f1, i2, f2, Tf = task
    from k_vv_mm import k_vv_mm

    k = float(k_vv_mm(_M1_RVV, _M2_RVV, i1, f1, i2, f2, Tf))
    return {"T": float(Tf), "i1": i1, "f1": f1, "i2": i2, "f2": f2, "k_VV": k}


quad_rvv = sorted(transitions_for_r_vv_fast(V_MAX))
tasks_rvv = [
    (i1, f1, i2, f2, float(T))
    for T in TEMPERATURES_K
    for i1, f1, i2, f2 in quad_rvv
]


def _rvv_executor(max_workers: int | None, m1, m2):
    exec_kw = {"initializer": _rvv_worker_init, "initargs": (m1, m2)}
    if sys.platform == "darwin" or sys.platform.startswith("linux"):
        try:
            ctx = mp.get_context("fork")
            return ProcessPoolExecutor(
                max_workers=max_workers, mp_context=ctx, **exec_kw
            )
        except ValueError:
            pass
    return ProcessPoolExecutor(max_workers=max_workers, **exec_kw)


t0_rvv = time.time()
rows_rvv: list[dict] = []
with _rvv_executor(RVV_FAST_WORKERS, M1, M2) as ex:
    fmap = {ex.submit(_compute_rvv_row, t): t for t in tasks_rvv}
    for fut in as_completed(fmap):
        row = fut.result()
        i1, f1, i2, f2 = row["i1"], row["f1"], row["i2"], row["f2"]
        print(
            f"готово: T={row['T']:.6g} K, i1={i1} f1={f1} i2={i2} f2={f2}, k_VV={row['k_VV']:.6g}",
            flush=True,
        )
        rows_rvv.append(row)

df_dataset_rvv_fast = pd.DataFrame(rows_rvv)
df_dataset_rvv_fast = df_dataset_rvv_fast.sort_values(
    ["T", "i1", "f1", "i2", "f2"], kind="mergesort"
).reset_index(drop=True)
elapsed_rvv = time.time() - t0_rvv
print(
    f"R_VV_fast: уникальных переходов {len(quad_rvv)}, строк {len(df_dataset_rvv_fast)}, "
    f"время {elapsed_rvv:.2f} s, workers={RVV_FAST_WORKERS!r}"
)
df_dataset_rvv_fast.to_csv(DATASET_RVV_FAST_PATH, index=False)
print(f"Сохранено: {DATASET_RVV_FAST_PATH.resolve()}")
df_dataset_rvv_fast.head()

## Регрессия

In [ ]:
def mape_loss(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100.0)


def k_model_poly(T, *coeffs):
    x = T ** (-1 / 3)
    return sum(c * x**k for k, c in enumerate(coeffs))


def k_predict_from_ln_poly(T_arr: np.ndarray, coeffs: np.ndarray) -> np.ndarray:
    T_arr = np.asarray(T_arr, dtype=float)
    return np.exp(k_model_poly(T_arr, *coeffs))


def fit_all_transitions(
    df: pd.DataFrame, poly_term_counts=POLY_TERM_COUNTS
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    df — длинный формат.
    Возвращает таблицу коэффициентов и wide-таблицу предсказанных k для MAPE-проверки.
    """
    df["transition"] = list(zip(df["i1"], df["f1"], df["i2"], df["f2"]))
    wide = df.pivot(index="T", columns="transition", values="k_VV")

    coeffs_rows = []
    pred_wide = pd.DataFrame(index=wide.index)

    T_col = wide.index.astype(float).values

    for col in wide.columns:
        y = wide[col].values.astype(float)

        best_mape = np.inf
        best_coeffs = None

        y_log = np.log(y)
        norm_factor = np.abs(y_log).max()
        y_norm = y_log / norm_factor

        for nc in poly_term_counts:
            params_norm, _ = curve_fit(
                k_model_poly,
                T_col,
                y_norm,
                p0=np.ones(nc),
                method="lm",
            )
            coeffs = params_norm * norm_factor
            k_pred = np.exp(k_model_poly(T_col, *coeffs))
            mm = mape_loss(y, k_pred)
            if mm < best_mape:
                best_mape = mm
                best_coeffs = coeffs

        assert best_coeffs is not None
        i1, f1, i2, f2 = col
        coeffs_rows.append(
            {"i1": i1, "f1": f1, "i2": i2, "f2": f2, "_mape": best_mape, "coeffs": best_coeffs}
        )
        pred_wide[col] = k_predict_from_ln_poly(T_col, best_coeffs)

    coeff_len = max(len(r["coeffs"]) for r in coeffs_rows)
    out = pd.DataFrame(
        [
            {
                "i1": r["i1"],
                "f1": r["f1"],
                "i2": r["i2"],
                "f2": r["f2"],
                **{f"a_{j}": r["coeffs"][j] if j < len(r["coeffs"]) else np.nan for j in range(coeff_len)},
            }
            for r in coeffs_rows
        ]
    )
    # Диагностический столбец (не входит в ваш список, но полезен; можно удалить перед экспортом)
    out.insert(5, "mape_pct", [r["_mape"] for r in coeffs_rows])
    return out, pred_wide

# Чтение из файла
df_dataset_rvv_fast = pd.read_csv(DATASET_RVV_FAST_PATH)

# Приводим типы данных как при сохранении
df_dataset_rvv_fast['T'] = df_dataset_rvv_fast['T'].astype(float)
df_dataset_rvv_fast['i1'] = df_dataset_rvv_fast['i1'].astype(int)
df_dataset_rvv_fast['f1'] = df_dataset_rvv_fast['f1'].astype(int)
df_dataset_rvv_fast['i2'] = df_dataset_rvv_fast['i2'].astype(int)
df_dataset_rvv_fast['f2'] = df_dataset_rvv_fast['f2'].astype(int)
df_dataset_rvv_fast['k_VV'] = df_dataset_rvv_fast['k_VV'].astype(float)

# Проверка
print(f"Загружено строк: {len(df_dataset_rvv_fast)}")
print(df_dataset_rvv_fast.head())
print(df_dataset_rvv_fast.dtypes)


df_coeffs, df_pred_wide = fit_all_transitions(df_dataset_rvv_fast.copy())
df_coeffs.drop(columns=["mape_pct"]).to_csv(COEFS_PATH, index=False)
print(f"Сохранено: {COEFS_PATH.resolve()}")
df_coeffs.head()

In [ ]:
# ============== ТВОИ ФУНКЦИИ (БЕЗ ИЗМЕНЕНИЙ) ==============
def mape_loss(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100.0)


def k_model_poly(T, *coeffs):
    x = np.power(np.asarray(T, dtype=float), -1.0 / 3.0)
    out = np.zeros_like(x, dtype=float)
    for k, c in enumerate(coeffs):
        out += float(c) * np.power(x, k)
    return out


def fit_poly(df: pd.DataFrame):
    """Как в `regression.ipynb`: для каждой числовой колонки после `T` подбирает степень 3…6 параметров LM + MAPE."""
    params_arr = []
    pred_df = pd.DataFrame(index=df.index)
    for col_i in range(1, df.shape[1]):
        coeffs = None
        pred = np.zeros(len(df))
        best_mape = np.inf

        series = pd.to_numeric(df.iloc[:, col_i], errors="coerce")
        mask = np.isfinite(series.values) & (series.values > 0)
        T_col = pd.to_numeric(df.iloc[:, 0], errors="coerce")
        T_valid = T_col.values[mask]
        y_valid = series.values[mask]

        if len(T_valid) < 4:
            raise ValueError("Недостаточно положительных точек для полинома.")

        y_log = np.log(y_valid)
        norm_factor = float(np.max(np.abs(y_log)))
        if norm_factor == 0:
            norm_factor = 1.0
        y_norm = y_log / norm_factor

        for coeff_num in range(4, 8):
            p0 = np.ones(coeff_num)
            params, _ = curve_fit(
                k_model_poly,
                T_valid,
                y_norm,
                p0=p0,
                method="lm",
                maxfev=40000,
            )

            params1 = params * norm_factor
            y_pred_log_full = k_model_poly(T_col.values.astype(float), *params1)
            k_pred_full = np.exp(y_pred_log_full)

            valid_pred = np.isfinite(k_pred_full) & mask
            if not np.any(valid_pred):
                continue
            mape_poly = mape_loss(series.values[valid_pred], k_pred_full[valid_pred])

            if mape_poly < best_mape:
                best_mape = mape_poly
                coeffs = params1.astype(float)
                pred = k_pred_full

        if coeffs is None:
            raise RuntimeError("curve_fit не сработала ни для одного порядка.")

        params_arr.append(coeffs)
        pred_df[col_i - 1] = pred

    return params_arr, pred_df


# ============== ЗАГРУЗКА И ПРЕОБРАЗОВАНИЕ ==============
print("Загрузка данных...")
df_long = pd.read_csv(DATASET_RVV_FAST_PATH)

# Приводим типы
df_long['T'] = df_long['T'].astype(float)
df_long['i1'] = df_long['i1'].astype(int)
df_long['f1'] = df_long['f1'].astype(int)
df_long['i2'] = df_long['i2'].astype(int)
df_long['f2'] = df_long['f2'].astype(int)
df_long['k_VV'] = df_long['k_VV'].astype(float)

print(f"Загружено строк: {len(df_long)}")

# Преобразуем в wide формат (как в твоем оригинале)
print("Преобразование в wide формат...")
df_long['transition'] = df_long.apply(
    lambda row: f"{row['i1']}_{row['f1']}_{row['i2']}_{row['f2']}", axis=1
)

# Pivot
df_wide = df_long.pivot(index='T', columns='transition', values='k_VV')

print(f"Wide формат: {df_wide.shape[0]} температур × {df_wide.shape[1]} переходов")

# Сбрасываем индекс чтобы T стала колонкой
df_wide = df_wide.reset_index()

# Переставляем колонки: первая T, остальные - переходы
cols = ['T'] + [c for c in df_wide.columns if c != 'T']
df_wide = df_wide[cols]

print(f"Готово для fit_poly: {df_wide.shape}")

# ============== ЗАПУСК РЕГРЕССИИ ==============
print("\nЗапуск регрессии (это может занять время)...")
print("=" * 60)

try:
    params_arr, pred_df = fit_poly(df_wide)
    print(f"Регрессия завершена! Обработано переходов: {len(params_arr)}")

    # ============== СОХРАНЕНИЕ КОЭФФИЦИЕНТОВ (ИСПРАВЛЕННЫЙ) ==============
    # Получаем названия переходов из колонок (пропуская T)
    transitions = df_wide.columns[1:]

    coeffs_rows = []
    for col_name, coeffs in zip(transitions, params_arr):
        # Парсим переход - убираем возможные .0
        parts = col_name.split('_')
        # Очищаем от .0 и преобразуем в int
        i1 = int(float(parts[0]))
        f1 = int(float(parts[1]))
        i2 = int(float(parts[2]))
        f2 = int(float(parts[3]))

        row = {
            'i1': i1, 'f1': f1,
            'i2': i2, 'f2': f2
        }

        # Добавляем коэффициенты
        for j, c in enumerate(coeffs):
            row[f'a_{j}'] = c

        coeffs_rows.append(row)

    df_coeffs = pd.DataFrame(coeffs_rows)
    df_coeffs.to_csv(COEFS_PATH, index=False)

    print(f"\n✅ Сохранено: {COEFS_PATH.resolve()}")
    print(f"Всего переходов: {len(df_coeffs)}")
    print("\nПервые 5 результатов:")
    print(df_coeffs.head())

except Exception as e:
    print(f"Ошибка: {e}")
    import traceback

    traceback.print_exc()

При необходимости сохраните версию **с** столбцом `mape_pct` для отбора плохих переходов.

In [ ]:
df_coeffs_with_mape = df_coeffs.copy()
df_coeffs_with_mape.to_csv(
    WORKDIR / f"{PARTICLE1_NAME}_{PARTICLE2_NAME}_regression_coefs_with_mape.csv",
    index=False,
)

In [ ]:
print("fff")

## График: точный k_VV и регрессия для выбранного перехода

In [ ]:
PLOT_TRANSITION = (0, 1, 1, 0)

def row_coeffs_for_transition(
    df_coef: pd.DataFrame, i1: int, f1: int, i2: int, f2: int
) -> np.ndarray:
    mask = (
        (df_coef["i1"] == i1)
        & (df_coef["f1"] == f1)
        & (df_coef["i2"] == i2)
        & (df_coef["f2"] == f2)
    )
    if not mask.any():
        raise KeyError((i1, f1, i2, f2))
    row = df_coef.loc[mask].iloc[0]
    a_cols = [c for c in df_coef.columns if c.startswith("a_")]
    vals = [row[c] for c in sorted(a_cols, key=lambda s: int(s.split("_")[1]))]
    return np.array([v for v in vals if pd.notna(v)], dtype=float)


i1_p, f1_p, i2_p, f2_p = PLOT_TRANSITION

# Плотная сетка по T только для красивой кривой регрессии
T_fine = np.linspace(TEMPERATURES_K.min(), TEMPERATURES_K.max(), 20)


def _k_exact_fine_init(m1, m2, i1, f1, i2, f2) -> None:
    global _KM1, _KM2, _KI1, _KF1, _KI2, _KF2
    _KM1, _KM2 = m1, m2
    _KI1, _KF1, _KI2, _KF2 = i1, f1, i2, f2


def _k_exact_fine_one(Tf: float) -> float:
    from k_vv_mm import k_vv_mm

    return float(k_vv_mm(_KM1, _KM2, _KI1, _KF1, _KI2, _KF2, float(Tf)))


def _fine_k_executor(max_workers, m1, m2, i1, f1, i2, f2):
    exec_kw = {
        "initializer": _k_exact_fine_init,
        "initargs": (m1, m2, i1, f1, i2, f2),
    }
    if sys.platform == "darwin" or sys.platform.startswith("linux"):
        try:
            ctx = mp.get_context("fork")
            return ProcessPoolExecutor(
                max_workers=max_workers, mp_context=ctx, **exec_kw
            )
        except ValueError:
            pass
    return ProcessPoolExecutor(max_workers=max_workers, **exec_kw)


T_fine_list = [float(t) for t in T_fine]
with _fine_k_executor(RVV_FAST_WORKERS, M1, M2, i1_p, f1_p, i2_p, f2_p) as ex:
    k_exact_fine = np.array(list(ex.map(_k_exact_fine_one, T_fine_list)), dtype=float)

cfs = row_coeffs_for_transition(df_coeffs, i1_p, f1_p, i2_p, f2_p)
k_reg_fine = k_predict_from_ln_poly(T_fine, cfs)

# Точки сетки (исходный датасет)
msk = (
    (df_dataset_rvv_fast["i1"] == i1_p)
    & (df_dataset_rvv_fast["f1"] == f1_p)
    & (df_dataset_rvv_fast["i2"] == i2_p)
    & (df_dataset_rvv_fast["f2"] == f2_p)
)
dsub = df_dataset_rvv_fast.loc[msk].sort_values("T")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(dsub["T"], dsub["k_VV"], color="C0", s=36, label="k_vv_mm (узлы)", zorder=3)
ax.plot(T_fine, k_exact_fine, color="C0", alpha=0.5, lw=2, label="k_vv_mm (плотная сетка)")
ax.plot(T_fine, k_reg_fine, "--", color="C1", lw=2, label="регрессия")
ax.set_xlabel("T, K")
ax.set_ylabel("k_VV, см³/с")
ax.set_title(f"Переход (i1={i1_p},f1={f1_p}) + (i2={i2_p},f2={f2_p})")
ax.set_yscale("log")
ax.legend()
ax.grid(True, which="both", ls=":", alpha=0.5)
plt.tight_layout()
plt.show()